In [1]:
!pip install torch==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install numpy==1.26.4 scipy==1.12.0
!pip install recbole codecarbon kmeans_pytorch

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached recbole-1.2.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached codecarbon-3.2.7-py3-none-any.whl.metadata (9.7 kB)
  Using cached kmeans_pytorch-0.3-py3-none-any.whl.metadata (1.6 kB)
  Using cached colorlog-4.7.2-py2.py3-none-any.whl.metadata (9.9 kB)
  Using cached colorama-0.4.4-py2.py3-none-any.whl.metadata (14 kB)
  Using cached thop-0.1.1.post2209072238-py3-none-any.whl.metadata (2.7 kB)
  Using cached texttable-1.7.0-py2.py3-none-any.whl.metadata (9.8 kB)
INFO: pip is looking at multiple versions of recbole to determine which version is compatible with other requirements. This could take a while.
  Using cached recbole-1.2.0-py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.5/380.5 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━

In [1]:
import warnings
warnings.filterwarnings("ignore")


In [2]:
import pandas as pd
import torch
from logging import getLogger

from codecarbon import EmissionsTracker
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed, init_logger, get_model, get_trainer

dataset_name = "ml-100k"
device = "cuda" if torch.cuda.is_available() else "cpu"

base_config = {
    "seed": 2024,
    "reproducibility": True,
    "device": device,
    "metrics": ["NDCG", "Recall", "MRR", "Precision", "Hit", "MAP"],
    "topk": [5, 10],
    "valid_metric": "NDCG@10",
    "eval_args": {"split": {"RS": [0.8, 0.1, 0.1]}, "order": "RO", "mode": "full"},
}

loss_options = ["BPR", "CE"]
loss_models = {"SASRec", "GRU4Rec", "NextItNet", "Caser"}
EPOCHS = 1
STOPPING_STEP = 10
TRAIN_BATCH_SIZE = 2048
EMBEDDING_SIZE = 64
seed_variants = [2024]

def make_loss_configs(model_name, base_runs):
    configs = []
    for loss in loss_options:
        for idx, run in enumerate(base_runs, start=1):
            config = run.copy()
            config["loss_type"] = loss
            configs.append({
                "name": f"{model_name}_{loss}",
                "config": config,
                "loss": loss,
            })
    return configs

def make_noloss_configs(model_name, base_runs):
    configs = []
    for seed in seed_variants:
        for idx, run in enumerate(base_runs, start=1):
            config = run.copy()
            config["seed"] = seed
            configs.append({
                "name": f"{model_name}_s{seed}",
                "config": config,
                "loss": "N/A",
            })
    return configs

base_runs = {
    "SASRec": [
        {
            "epochs": EPOCHS,
            "stopping_step": STOPPING_STEP,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "embedding_size": EMBEDDING_SIZE,
        },
    ],
    "GRU4Rec": [
        {
            "epochs": EPOCHS,
            "stopping_step": STOPPING_STEP,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "embedding_size": EMBEDDING_SIZE,
        },
    ],
    "NextItNet": [
        {
            "epochs": EPOCHS,
            "stopping_step": STOPPING_STEP,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "embedding_size": EMBEDDING_SIZE,
        },
    ],
    "Caser": [
        {
            "epochs": EPOCHS,
            "stopping_step": STOPPING_STEP,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "embedding_size": EMBEDDING_SIZE,
        },
    ],
    "Pop": [
        {
            "epochs": EPOCHS,
            "stopping_step": STOPPING_STEP,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "embedding_size": EMBEDDING_SIZE,
        },
    ],
    "Random": [
        {
            "epochs": EPOCHS,
            "stopping_step": STOPPING_STEP,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "embedding_size": EMBEDDING_SIZE,
        },
    ],
}

benchmark_configs = {}
for model_name, runs in base_runs.items():
    if model_name in loss_models:
        benchmark_configs[model_name] = make_loss_configs(model_name, runs)
    else:
        benchmark_configs[model_name] = make_noloss_configs(model_name, runs)

In [3]:
import subprocess
import platform

def detectar_hardware():
    info = {}

    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        info["gpu"]     = gpu_name
        info["vram_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
        info["device"]  = "cuda"

        g = gpu_name.lower()
        if   "h100" in g:            hw_label = "H100_GPU"
        elif "a100" in g:            hw_label = "A100_GPU"
        elif "l4"   in g:            hw_label = "L4_GPU"
        elif "t4"   in g:            hw_label = "T4_GPU"
        elif "g4"   in g or "v100" in g: hw_label = "G4_GPU"
        else:                        hw_label = gpu_name.replace(" ", "_")
    else:
        info["gpu"]    = None
        info["device"] = "cpu"
        try:
            import torch_xla
            hw_label       = "TPU"
            info["device"] = "xla"
        except ImportError:
            hw_label = "CPU"

    try:
        raw = subprocess.check_output("lscpu | grep 'Model name'", shell=True).decode().strip()
        info["cpu"] = raw.split(":")[-1].strip()
    except Exception:
        info["cpu"] = platform.processor() or "desconocido"

    try:
        raw   = subprocess.check_output("free -g | grep Mem", shell=True).decode().strip()
        ram   = int(raw.split()[1])
        info["ram_gb"]  = ram
        info["high_ram"] = ram >= 50
        if info["high_ram"]:
            hw_label += "_HighRAM"
    except Exception:
        info["ram_gb"]  = None
        info["high_ram"] = False

    info["hw_label"] = hw_label
    return info


HW_INFO  = detectar_hardware()
HW_LABEL = HW_INFO["hw_label"]
DEVICE   = HW_INFO["device"]

print("-" * 46)
print(f"hardware : {HW_LABEL}")
print(f"device   : {DEVICE}")
print(f"cpu      : {HW_INFO.get('cpu', 'N/A')}")
print(f"ram      : {HW_INFO.get('ram_gb', 'N/A')} GB  (high_ram={HW_INFO.get('high_ram', False)})")
if HW_INFO.get("gpu"):
    print(f"gpu      : {HW_INFO['gpu']} ({HW_INFO.get('vram_gb', '?')} GB VRAM)")
print("-" * 46)

----------------------------------------------
hardware : A100_GPU_HighRAM
device   : cuda
cpu      : Intel(R) Xeon(R) CPU @ 2.20GHz
ram      : 83 GB  (high_ram=True)
gpu      : NVIDIA A100-SXM4-40GB (42.4 GB VRAM)
----------------------------------------------


In [ ]:
import os
import time

def _read_codecarbon_last_row(csv_path):
    if not os.path.exists(csv_path):
        return None
    df = pd.read_csv(csv_path)
    if df.empty:
        return None
    return df.iloc[-1]

def _run_with_config(model_name, config_dict,  timestamp, config_name=None,):
    config = Config(model=model_name, dataset=dataset_name, config_dict=config_dict)
    init_seed(config["seed"], config["reproducibility"])
    init_logger(config)
    logger = getLogger()
    logger.info(config)

    dataset = create_dataset(config)
    train_data, valid_data, test_data = data_preparation(config, dataset)
    model = get_model(config["model"])(config, train_data.dataset).to(config["device"])
    trainer = get_trainer(config["MODEL_TYPE"], config["model"])(config, model)

    output_dir = f"codecarbon/{HW_LABEL}"
    os.makedirs(output_dir, exist_ok=True)

    name_part = config_name if config_name else model_name
    run_id = f"{HW_LABEL}_{name_part}_{timestamp}"
    output_file = f"{run_id}.csv"
    csv_path = os.path.join(output_dir, output_file)

    tracker = EmissionsTracker(
        project_name=run_id,
        measure_power_secs=1,
        log_level="error",
        output_dir=output_dir,
        output_file=output_file,
        save_to_file=True,
    )
    tracker.start()
    trainer.fit(train_data, valid_data)
    test_result = trainer.evaluate(test_data)
    emissions_kg = tracker.stop()

    last_row = _read_codecarbon_last_row(csv_path)
    time_s = float(last_row["duration"]) if last_row is not None else 0.0

    epochs_to_converge = None
    if hasattr(trainer, "cur_epoch"):
        epochs_to_converge = int(trainer.cur_epoch) + 1
    return test_result, emissions_kg, time_s, epochs_to_converge

def run_experiment(model_name, config_overrides, timestamp, config_name=None):
    config_dict = {**base_config, **config_overrides}
    if model_name in loss_models:
        eval_args = config_dict.get("eval_args", {})
        eval_args = {**eval_args, "order": "TO"}
        config_dict["eval_args"] = eval_args

    loss_used = config_dict.get("loss_type")

    if loss_used == "CE":
        config_dict["train_neg_sample_args"] = None

    try:
        test_result, emissions_kg, time_s, epochs_to_converge = _run_with_config(model_name, config_dict,
timestamp, config_name)
    except ValueError as exc:
        if loss_used and "loss" in str(exc).lower():
            config_dict = {k: v for k, v in config_dict.items() if k != "loss_type"}
            loss_used = None

            if "train_neg_sample_args" in str(exc):
                config_dict["train_neg_sample_args"] = None

            test_result, emissions_kg, time_s, epochs_to_converge = _run_with_config(model_name, config_dict,
timestamp, config_name)
        else:
            raise

    return test_result, emissions_kg, time_s, epochs_to_converge, loss_used


In [ ]:
def smoke_test(model_name="SASRec", loss_type="BPR", epochs=1, stopping_step=1, train_batch_size=2048,
embedding_size=64):
    lote_timestamp = time.strftime("%Y%m%d_%H%M%S")
    override = {
        "epochs": epochs,
        "stopping_step": stopping_step,
        "train_batch_size": train_batch_size,
        "embedding_size": embedding_size,
    }
    if loss_type:
        override["loss_type"] = loss_type

    test_result, emissions_kg, time_s, epochs_to_converge, loss_used = run_experiment(
        model_name, override, lote_timestamp, f"{model_name}_{loss_type}_smoke"
    )
    return {
        "hardware": HW_LABEL,
        "model": model_name,
        "loss": loss_type,
        "loss_used": loss_used if loss_used else "N/A",
        "co2_kg": emissions_kg,
        "time_s": time_s,
        "epochs_to_converge": epochs_to_converge,
        "ndcg@10": test_result.get("ndcg@10", test_result.get("NDCG@10")),
    }

def run_smoke_benchmarks(benchmark_configs):
    results = []
    i = 0
    for model_name in benchmark_configs:
        if i == 1:
            break
        results.append(smoke_test(model_name=model_name, epochs=1))
        i += 1
    return pd.DataFrame(results).sort_values(["model", "loss"]).reset_index(drop=True)

smoke_df = run_smoke_benchmarks(benchmark_configs)
smoke_df

[codecarbon WARNING @ 15:07:30] Multiple instances of codecarbon are allowed to run at the same time.


,hardware,model,loss,loss_used,co2_kg,time_s,epochs_to_converge,ndcg@10
0,A100_GPU_HighRAM,SASRec,BPR,BPR,0.000065,4.611047,None,0.0103


In [6]:
lote_timestamp = time.strftime("%Y%m%d_%H%M%S")

print(f"Timestamp para esta ejecución: {lote_timestamp}")
print(f"Hardware detectado: {HW_LABEL} (device={DEVICE})")
print("-" * 46)
print("Configuraciones a ejecutar:")
for model_name, configs in benchmark_configs.items():
    for config_entry in configs:
        print(f"  - Modelo: {model_name}, Config: {config_entry['name']}, Loss: {config_entry.get('loss', 'N/A')}")
print("EPOCHS", EPOCHS)
print("STOPPING_STEP", STOPPING_STEP)
print("TRAIN_BATCH_SIZE", TRAIN_BATCH_SIZE)
print("EMBEDDING_SIZE", EMBEDDING_SIZE)
print("-" * 46)
print(f"Total de modelos: {len(benchmark_configs)}")
print(f"Total de configuraciones: {sum(len(configs) for configs in benchmark_configs.values())}")
print(f"Total de ejecuciones planeadas: {sum(len(configs) for configs in benchmark_configs.values())}")
print("Iniciando benchmarks...")


warnings.filterwarnings("ignore")

metric_keys = [
    "ndcg@5", "ndcg@10",
    "recall@5", "recall@10",
    "mrr@5", "mrr@10",
    "precision@5", "precision@10",
    "hit@5", "hit@10",
    "map@5", "map@10",
    "NDCG@5", "NDCG@10",
    "Recall@5", "Recall@10",
    "MRR@5", "MRR@10",
    "Precision@5", "Precision@10",
    "Hit@5", "Hit@10",
    "MAP@5", "MAP@10",
 ]
records = []

for model_name, configs in benchmark_configs.items():
    for config_entry in configs:
        config_name = config_entry["name"]
        config_overrides = config_entry["config"]
        loss_label = config_entry.get("loss", "N/A")
        test_result, emissions_kg, time_s, epochs_to_converge, loss_used = run_experiment(model_name, config_overrides, lote_timestamp, config_name)
        row = {
            "hardware": HW_LABEL,
            "model": model_name,
            "config": config_name,
            "loss": loss_label,
            "loss_used": loss_used if loss_used else "N/A",
            "co2_kg": emissions_kg,
            "time_s": time_s,
            "epochs_to_converge": epochs_to_converge,
        }
        for key in metric_keys:
            if key in test_result:
                row[key.lower()] = test_result[key]
        records.append(row)

results_df = pd.DataFrame(records)
results_df = results_df.sort_values(["model", "config"]).reset_index(drop=True)
results_df

import os
os.makedirs("resultados", exist_ok=True)
output_path = f"resultados/benchmark_{lote_timestamp}_{HW_LABEL}.csv"
results_df.to_csv(output_path, index=False)
print(f"Resultados guardados en: {output_path}")

Timestamp para esta ejecución: 20260605_150741
Hardware detectado: A100_GPU_HighRAM (device=cuda)
----------------------------------------------
Configuraciones a ejecutar:
  - Modelo: SASRec, Config: SASRec_BPR, Loss: BPR
  - Modelo: SASRec, Config: SASRec_CE, Loss: CE
  - Modelo: GRU4Rec, Config: GRU4Rec_BPR, Loss: BPR
  - Modelo: GRU4Rec, Config: GRU4Rec_CE, Loss: CE
  - Modelo: NextItNet, Config: NextItNet_BPR, Loss: BPR
  - Modelo: NextItNet, Config: NextItNet_CE, Loss: CE
  - Modelo: Caser, Config: Caser_BPR, Loss: BPR
  - Modelo: Caser, Config: Caser_CE, Loss: CE
  - Modelo: Pop, Config: Pop_s2024, Loss: N/A
  - Modelo: Random, Config: Random_s2024, Loss: N/A
EPOCHS 1
STOPPING_STEP 10
TRAIN_BATCH_SIZE 2048
EMBEDDING_SIZE 64
----------------------------------------------
Total de modelos: 6
Total de configuraciones: 10
Total de ejecuciones planeadas: 10
Iniciando benchmarks...


Resultados guardados en: resultados/benchmark_20260605_150741_A100_GPU_HighRAM.csv
